In [1]:
import os

# ---- NB06 convention: first cell always sets env vars + device ----
# M1 / libomp conflicts (project notes)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("Device:", device)

PyTorch: 2.10.0
MPS available: True
Device: mps


# 07 — Subtype flow analysis (Phase 3)

Goal: load the already-trained BRCA SoftPermMix model and compute **per-PAM50-subtype** 4×4 cross-omics flow matrices on the *test split*, then render a 5-panel heatmap figure (LumA, LumB, HER2, Basal, Normal).

Notes:
- No retraining.
- GS-BRCA `Original/` CSVs are stored as **features × patients** and must be transposed on load.
- The label CSV must be loaded **without** `index_col=0` and aligned positionally.
- Use ANOVA `SelectKBest(k=50)` per omics (M1 memory safe), then `StandardScaler` fit on train only.


## 0. Imports and utilities

In [2]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sns.set(style="white", context="notebook")


def seed_all(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def mps_cleanup():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


def find_one_csv(root: Path, include_any=(), include_all=(), exclude_any=()):
    """Heuristic helper to locate files despite minor naming differences."""
    root = Path(root)
    cands = list(root.glob("*.csv"))

    def ok(p: Path):
        name = p.name.lower()
        if include_any and not any(k.lower() in name for k in include_any):
            return False
        if include_all and not all(k.lower() in name for k in include_all):
            return False
        if exclude_any and any(k.lower() in name for k in exclude_any):
            return False
        return True

    cands = [p for p in cands if ok(p)]
    if len(cands) == 0:
        raise FileNotFoundError(
            f"No CSV match in {root} for include_any={include_any}, include_all={include_all}, exclude_any={exclude_any}"
        )
    cands = sorted(cands, key=lambda p: (len(p.name), p.name))
    return cands[0]


seed_all(42)

## 0.1 Paths and config

In [3]:
HOME = Path.home()

CMOB_DIR = HOME / "CMOB"
FIG_DIR = CMOB_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = HOME / "Cancer-Multi-Omics-Benchmark" / "Main_Dataset" / "Classification_datasets" / "GS-BRCA" / "Original"

# Checkpoint name is referenced with slightly different spellings; accept both.
CKPT_CANDS = [
    CMOB_DIR / "model_mix_brca.pt",
    CMOB_DIR / "modelmixbrca.pt",
    CMOB_DIR / "model_mix_brca.pth",
    CMOB_DIR / "modelmixbrca.pth",
]

LATENT_DIM = 64
K_PERMS = 4
N_CLASSES = 5
K_BEST = 50

# Fixed plot order
SUBTYPE_ORDER = ["LumA", "LumB", "HER2", "Basal", "Normal"]

print("DATA_DIR:", DATA_DIR)
print("FIG_DIR:", FIG_DIR)

DATA_DIR: /Users/wardabdelhafez/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Original
FIG_DIR: /Users/wardabdelhafez/CMOB/figures


## 1. Load GS-BRCA (Original) and preprocess (NB06-compatible)

Key quirks:
- Omics matrices must be loaded with `index_col=0` then transposed.
- Labels must be loaded **without** `index_col=0`.
- Alignment is positional (`reset_index(drop=True)`).


In [4]:
mrna_path  = find_one_csv(DATA_DIR, include_any=("mrna",))
mirna_path = find_one_csv(DATA_DIR, include_any=("mirna",))
methy_path = find_one_csv(DATA_DIR, include_any=("methy", "methyl"))
cnv_path   = find_one_csv(DATA_DIR, include_any=("cnv",))

# Labels: try a few common patterns, else fallback to "the remaining CSV".
try:
    labels_path = find_one_csv(DATA_DIR, include_any=("label", "pam50", "subtype", "y"), exclude_any=("mrna", "mirna", "methy", "cnv"))
except Exception:
    all_csvs = list(Path(DATA_DIR).glob("*.csv"))
    labels_path = [p for p in all_csvs if p not in {mrna_path, mirna_path, methy_path, cnv_path}][0]

print("mRNA :", mrna_path.name)
print("miRNA:", mirna_path.name)
print("Methy:", methy_path.name)
print("CNV  :", cnv_path.name)
print("Y    :", labels_path.name)


def load_omics_matrix(path: Path) -> pd.DataFrame:
    # Stored as features x patients → transpose to patients x features
    df = pd.read_csv(path, index_col=0).T
    df = df.reset_index(drop=True)
    return df


X_mrna  = load_omics_matrix(mrna_path)
X_mirna = load_omics_matrix(mirna_path)
X_methy = load_omics_matrix(methy_path)
X_cnv   = load_omics_matrix(cnv_path)

# Labels (positional)
y_df = pd.read_csv(labels_path)
if y_df.shape[1] == 1:
    y_raw = y_df.iloc[:, 0]
else:
    y_raw = y_df.dropna(axis=1, how="all").iloc[:, 0]
y_raw = y_raw.reset_index(drop=True)

n = len(y_raw)
assert X_mrna.shape[0] == n == X_mirna.shape[0] == X_methy.shape[0] == X_cnv.shape[0], "Patient counts must match across omics + labels"

print("Patients:", n)
print("Raw feature dims:", {
    "mRNA": X_mrna.shape[1],
    "miRNA": X_mirna.shape[1],
    "Methy": X_methy.shape[1],
    "CNV": X_cnv.shape[1]
})
print("Labels dtype:", y_raw.dtype)
print("Label counts:")
print(pd.Series(y_raw).value_counts())

mps_cleanup()

mRNA : BRCA_mRNA.csv
miRNA: BRCA_miRNA.csv
Methy: BRCA_Methy.csv
CNV  : BRCA_CNV.csv
Y    : BRCA_label_num.csv
Patients: 671
Raw feature dims: {'mRNA': 18206, 'miRNA': 368, 'Methy': 19049, 'CNV': 19568}
Labels dtype: int64
Label counts:
Label
0    353
2    132
4    113
1     42
3     31
Name: count, dtype: int64


## 1.1 Encode labels, split (70/15/15), ANOVA SelectKBest, StandardScaler

This matches the Phase 2 setup: ANOVA top-50 per block (200 total) and stratified split with seed=42.


In [5]:
# Map labels → 0..4 using SUBTYPE_ORDER
if (y_raw.dtype == "object") or isinstance(y_raw.iloc[0], str):
    y_str = y_raw.astype(str)
    # mild normalization
    y_str = y_str.str.replace("Luminal A", "LumA", regex=False)
    y_str = y_str.str.replace("Luminal B", "LumB", regex=False)
    y_str = y_str.str.replace("Her2", "HER2", regex=False)
    y_str = y_str.str.replace("Her-2", "HER2", regex=False)
    y_str = y_str.str.replace("Basal-like", "Basal", regex=False)

    bad = sorted(set(y_str.unique()) - set(SUBTYPE_ORDER))
    if bad:
        raise ValueError(f"Unexpected subtype labels: {bad}. Expected subset of {SUBTYPE_ORDER}")

    y = y_str.map({k: i for i, k in enumerate(SUBTYPE_ORDER)}).to_numpy(dtype=np.int64)
else:
    y = y_raw.to_numpy(dtype=np.int64)

# Stratified 70/15/15 with seed=42
idx_all = np.arange(n)
idx_tr, idx_tmp, y_tr, y_tmp = train_test_split(
    idx_all, y, test_size=0.30, random_state=42, stratify=y
)
idx_va, idx_te, y_va, y_te = train_test_split(
    idx_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

print("Split sizes:", {"train": len(idx_tr), "val": len(idx_va), "test": len(idx_te)})
print("Test counts:", dict(zip(*np.unique(y_te, return_counts=True))))


def fit_select_and_scale(X_train, X_val, X_test, y_train, k=50):
    selector = SelectKBest(score_func=f_classif, k=k)
    Xtr = selector.fit_transform(X_train, y_train)
    Xva = selector.transform(X_val)
    Xte = selector.transform(X_test)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xva = scaler.transform(Xva)
    Xte = scaler.transform(Xte)
    return Xtr, Xva, Xte, selector, scaler


Xm_tr,  Xm_va,  Xm_te,  sel_mrna,  sc_mrna  = fit_select_and_scale(X_mrna.iloc[idx_tr],  X_mrna.iloc[idx_va],  X_mrna.iloc[idx_te],  y_tr, k=K_BEST)
Xmi_tr, Xmi_va, Xmi_te, sel_mirna, sc_mirna = fit_select_and_scale(X_mirna.iloc[idx_tr], X_mirna.iloc[idx_va], X_mirna.iloc[idx_te], y_tr, k=K_BEST)
Xme_tr, Xme_va, Xme_te, sel_methy, sc_methy = fit_select_and_scale(X_methy.iloc[idx_tr], X_methy.iloc[idx_va], X_methy.iloc[idx_te], y_tr, k=K_BEST)
Xc_tr,  Xc_va,  Xc_te,  sel_cnv,   sc_cnv   = fit_select_and_scale(X_cnv.iloc[idx_tr],   X_cnv.iloc[idx_va],   X_cnv.iloc[idx_te],  y_tr, k=K_BEST)

print("Post-ANOVA dims:", {
    "mRNA": Xm_te.shape,
    "miRNA": Xmi_te.shape,
    "Methy": Xme_te.shape,
    "CNV": Xc_te.shape,
})

mps_cleanup()

Split sizes: {'train': 469, 'val': 101, 'test': 101}
Test counts: {np.int64(0): np.int64(53), np.int64(1): np.int64(7), np.int64(2): np.int64(20), np.int64(3): np.int64(4), np.int64(4): np.int64(17)}
Post-ANOVA dims: {'mRNA': (101, 50), 'miRNA': (101, 50), 'Methy': (101, 50), 'CNV': (101, 50)}


## 2. Model definition (must match training)

Important MPS rule from project debugging: after every standalone `nn.Module` instantiation, call `.to(device)`.


In [8]:
class SoftPermutationMix(nn.Module):
    def __init__(self, dim: int, K: int = 4):
        super().__init__()
        self.dim = dim
        self.K = K
        self.alpha_logits = nn.Parameter(torch.zeros(K))
        perms = [torch.eye(dim)[torch.randperm(dim)] for _ in range(K)]
        self.register_buffer("perms", torch.stack(perms))

    def get_mixing_matrix(self) -> torch.Tensor:
        alpha = torch.softmax(self.alpha_logits, dim=0)
        return torch.einsum("k,kij->ij", alpha, self.perms)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.get_mixing_matrix().T

    def get_alpha(self) -> np.ndarray:
        return torch.softmax(self.alpha_logits, dim=0).detach().cpu().numpy()


class MultiOmicsNet(nn.Module):
    def __init__(
        self,
        n_mrna: int,
        n_mirna: int,
        n_methy: int,
        n_cnv: int,
        latent_dim: int = 64,
        n_classes: int = 5,
        use_mix: bool = True,
        K: int = 4,
    ):
        super().__init__()
        self.use_mix = use_mix
        self.latent_dim = latent_dim

        def encoder(in_dim: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Linear(in_dim, latent_dim),
                nn.LayerNorm(latent_dim),
                nn.GELU(),
                nn.Dropout(0.3),
            )

        self.enc_mrna  = encoder(n_mrna)
        self.enc_mirna = encoder(n_mirna)
        self.enc_methy = encoder(n_methy)
        self.enc_cnv   = encoder(n_cnv)

        fused_dim = latent_dim * 4  # 256 with latent_dim=64

        if use_mix:
            self.mixer = SoftPermutationMix(fused_dim, K=K)

        self.head = nn.Sequential(
            nn.Linear(fused_dim, fused_dim),
            nn.LayerNorm(fused_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(fused_dim, n_classes),
        )

    def forward(self, x_mrna, x_mirna, x_methy, x_cnv) -> torch.Tensor:
        z = torch.cat([
            self.enc_mrna(x_mrna),
            self.enc_mirna(x_mirna),
            self.enc_methy(x_methy),
            self.enc_cnv(x_cnv),
        ], dim=1)
        if self.use_mix:
            z = self.mixer(z)
        return self.head(z)

    def get_mixing_matrix(self):
        if self.use_mix:
            return self.mixer.get_mixing_matrix().detach().cpu().numpy()
        return None

    def get_alpha(self):
        if self.use_mix:
            return self.mixer.get_alpha()
        return None


class BRCAOmicsDataset(Dataset):
    def __init__(self, Xm, Xmi, Xme, Xc, y):
        self.Xm  = torch.tensor(Xm,  dtype=torch.float32)
        self.Xmi = torch.tensor(Xmi, dtype=torch.float32)
        self.Xme = torch.tensor(Xme, dtype=torch.float32)
        self.Xc  = torch.tensor(Xc,  dtype=torch.float32)
        self.y   = torch.tensor(y,   dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.Xm[i], self.Xmi[i], self.Xme[i], self.Xc[i], self.y[i]

## 2.1 Load the already-trained BRCA mixer checkpoint

This notebook only *loads* the model and runs inference + flow extraction.


In [10]:
ckpt_path = None
for p in CKPT_CANDS:
    if p.exists():
        ckpt_path = p
        break
if ckpt_path is None:
    raise FileNotFoundError(f"Could not find checkpoint in: {[str(p) for p in CKPT_CANDS]}")

print("Loading checkpoint:", ckpt_path)

# Instantiate with BRCA-selected dims (50 per omics), latent=64, fused=256, K=4
model = MultiOmicsNet(
    n_mrna=K_BEST,
    n_mirna=K_BEST,
    n_methy=K_BEST,
    n_cnv=K_BEST,
    latent_dim=LATENT_DIM,
    n_classes=N_CLASSES,
    use_mix=True,
    K=K_PERMS,
).to(device)

ckpt = torch.load(ckpt_path, map_location=device)

if isinstance(ckpt, nn.Module):
    model = ckpt.to(device)
    print("Checkpoint contained full nn.Module")
else:
    state_dict = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    print("Loaded state_dict")
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    if missing[:5] or unexpected[:5]:
        print("Example missing:", missing[:5])
        print("Example unexpected:", unexpected[:5])

model.eval()
mps_cleanup()

with torch.no_grad():
    alpha = model.mixer.get_alpha()                          # already returns np.ndarray
    D = model.mixer.get_mixing_matrix().detach().cpu().numpy()  # returns tensor, needs conversion

print("alpha:", np.round(alpha, 6), "sum:", float(alpha.sum()))
print("D row-sum mean:", float(D.sum(axis=1).mean()), "col-sum mean:", float(D.sum(axis=0).mean()))

# ---- Sanity: alpha must NOT be uniform (if it is, key mismatch or wrong checkpoint) ----
assert abs(alpha.max() - alpha.min()) > 0.01, (
    f"Alpha looks uniform after load ({np.round(alpha, 4)}) — "
    f"possible key mismatch or wrong checkpoint. "
    f"Expected approx [0.246, 0.257, 0.225, 0.272] from NB06."
)
print("✓ Alpha non-uniform confirmed — checkpoint loaded correctly.")

Loading checkpoint: /Users/wardabdelhafez/CMOB/model_mix_brca.pt
Loaded state_dict
Missing keys: 0
Unexpected keys: 0
alpha: [0.245974 0.256707 0.225194 0.272124] sum: 1.0
D row-sum mean: 1.0 col-sum mean: 1.0
✓ Alpha non-uniform confirmed — checkpoint loaded correctly.


## 3. Flow extraction per subtype

Two related quantities:

1. **Raw mixer block flow**: summarize the fixed 256×256 mixing matrix \(D\) into a 4×4 matrix by averaging sub-block weights.

2. **Effective flow (recommended)**: weight \(D\) by each sample's pre-mixer latent magnitudes (captured via a forward-pre-hook on the mixer). This is what can vary by subtype even if \(D\) itself is fixed.


## 3.0 Scientific note: what "per-subtype flow" actually measures

`SoftPermutationMix` learns a **single fixed** 256×256 matrix D shared
across all samples. D does not condition on the input — it is the same
regardless of which patient or subtype is being processed.

What varies per subtype is **h**: the pre-mixer fused latent vector,
which is driven by each subtype's distinct omics signature (e.g. HER2
amplification inflating the CNV encoder output, LumA methylation
patterns inflating the Methy encoder output).

The per-subtype flow matrix therefore answers:

> *Given the fixed routing structure D learned during BRCA training,
> how much of each subtype's latent activation energy flows between
> omics blocks?*

This is **activation-weighted engagement** of the routing matrix, not
a dynamic or sample-adaptive router. The biological signal comes from
h, not from D changing. This is still a meaningful interpretability
result — it reveals which omics blocks are *actively used* by each
subtype given the shared routing structure.

In [11]:
test_ds = BRCAOmicsDataset(Xm_te, Xmi_te, Xme_te, Xc_te, y_te)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)

# Capture mixer input h (pre-mix fused latent) using a forward-pre-hook
cache = {}

def pre_hook(module, inputs):
    cache["h"] = inputs[0].detach()  # (B, 256)

hook_handle = model.mixer.register_forward_pre_hook(pre_hook)

latent = LATENT_DIM
blocks = {
    "mRNA": slice(0 * latent, 1 * latent),
    "miRNA": slice(1 * latent, 2 * latent),
    "Methy": slice(2 * latent, 3 * latent),
    "CNV": slice(3 * latent, 4 * latent),
}
block_names = list(blocks.keys())

with torch.no_grad():
    D_torch = model.mixer.get_mixing_matrix()  # (256,256)


def block_average_D(D_np: np.ndarray) -> np.ndarray:
    """Average each 64×64 sub-block of D into a 4×4 matrix."""
    out = np.zeros((4, 4), dtype=np.float32)
    for si, sname in enumerate(block_names):
        s = blocks[sname]
        for ti, tname in enumerate(block_names):
            t = blocks[tname]
            out[si, ti] = float(D_np[t, s].mean())
    return out


def effective_flow_batch(h_batch: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    """Compute per-sample normalized effective flow: (B, 4, 4).
    
    Each source block is L2-normalized before weighting by D, so the
    result is a relative routing fraction rather than raw activation 
    magnitude. This makes cross-subtype comparisons scale-invariant.
    """
    B = h_batch.shape[0]
    abs_h = h_batch.abs()
    out = torch.zeros((B, 4, 4), device=h_batch.device, dtype=h_batch.dtype)

    for si, sname in enumerate(block_names):
        s = blocks[sname]
        hs = abs_h[:, s]                                          # (B, 64)
        source_norm = hs.norm(dim=1, keepdim=True).clamp(min=1e-8) # (B, 1)
        hs_normed = hs / source_norm                              # unit vectors (B, 64)
        for ti, tname in enumerate(block_names):
            t = blocks[tname]
            W = D[t, s]                                           # (64, 64)
            tmp = hs_normed @ W.T                                 # (B, 64)
            out[:, si, ti] = tmp.mean(dim=1)
    return out


# Accumulate per subtype
flows_by_subtype = {k: [] for k in SUBTYPE_ORDER}
counts_by_subtype = {k: 0 for k in SUBTYPE_ORDER}

correct = 0
total = 0

with torch.no_grad():
    for Xm_b, Xmi_b, Xme_b, Xc_b, y_b in test_loader:
        Xm_b = Xm_b.to(device)
        Xmi_b = Xmi_b.to(device)
        Xme_b = Xme_b.to(device)
        Xc_b = Xc_b.to(device)
        y_b = y_b.to(device)

        logits = model(Xm_b, Xmi_b, Xme_b, Xc_b)
        preds = logits.argmax(dim=1)

        correct += int((preds == y_b).sum().item())
        total += int(y_b.numel())

        h_b = cache["h"]
        eff = effective_flow_batch(h_b, D_torch)  # (B,4,4)

        for cls_idx, cls_name in enumerate(SUBTYPE_ORDER):
            mask = (y_b == cls_idx)
            if mask.any():
                flows_by_subtype[cls_name].append(eff[mask].detach().cpu().numpy())
                counts_by_subtype[cls_name] += int(mask.sum().item())

test_acc = correct / max(total, 1)
print(f"Test accuracy (sanity): {test_acc:.4f}")
print("Subtype counts (test):", counts_by_subtype)

# Reduce to mean matrices
mean_eff_flow = {}
for st in SUBTYPE_ORDER:
    if counts_by_subtype[st] == 0:
        mean_eff_flow[st] = np.full((4, 4), np.nan, dtype=np.float32)
        continue
    arr = np.concatenate(flows_by_subtype[st], axis=0)  # (n_st,4,4)
    mean_eff_flow[st] = arr.mean(axis=0)

# Also compute the raw block-averaged D for reference (same for all subtypes)
raw_D_block = block_average_D(D)
print("Raw D (block-averaged 4x4):")
print(pd.DataFrame(raw_D_block, index=block_names, columns=block_names).round(6))

hook_handle.remove()
mps_cleanup()

Test accuracy (sanity): 0.8020
Subtype counts (test): {'LumA': 53, 'LumB': 7, 'HER2': 20, 'Basal': 4, 'Normal': 17}
Raw D (block-averaged 4x4):
           mRNA     miRNA     Methy       CNV
mRNA   0.003881  0.003467  0.004507  0.003771
miRNA  0.003771  0.004325  0.003973  0.003556
Methy  0.004134  0.003593  0.003492  0.004406
CNV    0.003840  0.004240  0.003653  0.003892


## 4. Plot and save figures

Outputs (saved to `~/CMOB/figures/`):
- `07subtypeflowluma.png`, `07subtypeflowlumb.png`, `07subtypeflowher2.png`, `07subtypeflowbasal.png`, `07subtypeflownormal.png`
- `07subtypeflowpanel.png`
- `07subtypeflow_matrices.npz` (numerical matrices + counts)


In [12]:
vals = np.stack([mean_eff_flow[st] for st in SUBTYPE_ORDER], axis=0)
vmin = float(np.nanmin(vals))
vmax = float(np.nanmax(vals))


def plot_one(ax, mat, title, show_cbar=False):
    df = pd.DataFrame(mat, index=block_names, columns=block_names)
    sns.heatmap(
        df,
        ax=ax,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        square=True,
        cbar=show_cbar,
        annot=True,
        fmt=".4f",
        linewidths=0.5,
        linecolor="white",
    )
    ax.set_title(title)
    ax.set_xlabel("To (target block)")
    ax.set_ylabel("From (source block)")


name_map = {
    "LumA": "luma",
    "LumB": "lumb",
    "HER2": "her2",
    "Basal": "basal",
    "Normal": "normal",
}

# Individual subtype figures
for st in SUBTYPE_ORDER:
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    plot_one(ax, mean_eff_flow[st], f"{st} (n={counts_by_subtype[st]})", show_cbar=True)
    fig.tight_layout()
    outpath = FIG_DIR / f"07subtypeflow{name_map[st]}.png"
    fig.savefig(outpath, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", outpath)

# Combined 5-panel
fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)
for i, st in enumerate(SUBTYPE_ORDER):
    plot_one(axes[i], mean_eff_flow[st], f"{st} (n={counts_by_subtype[st]})", show_cbar=(i == 4))

panel_path = FIG_DIR / "07subtypeflowpanel.png"
fig.savefig(panel_path, dpi=260, bbox_inches="tight")
plt.close(fig)
print("Saved:", panel_path)

# Save numeric results
npz_path = FIG_DIR / "07subtypeflow_matrices.npz"
np.savez_compressed(
    npz_path,
    subtype_order=np.array(SUBTYPE_ORDER),
    block_names=np.array(block_names),
    counts=json.dumps(counts_by_subtype),
    raw_D_block=raw_D_block,
    **{f"eff_flow_{st}": mean_eff_flow[st] for st in SUBTYPE_ORDER},
)
print("Saved:", npz_path)

Saved: /Users/wardabdelhafez/CMOB/figures/07subtypeflowluma.png
Saved: /Users/wardabdelhafez/CMOB/figures/07subtypeflowlumb.png
Saved: /Users/wardabdelhafez/CMOB/figures/07subtypeflowher2.png
Saved: /Users/wardabdelhafez/CMOB/figures/07subtypeflowbasal.png
Saved: /Users/wardabdelhafez/CMOB/figures/07subtypeflownormal.png
Saved: /Users/wardabdelhafez/CMOB/figures/07subtypeflowpanel.png
Saved: /Users/wardabdelhafez/CMOB/figures/07subtypeflow_matrices.npz
